In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import get_cosine_schedule_with_warmup
from dataclasses import dataclass
from torch.utils.tensorboard import SummaryWriter
import datetime

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
@dataclass
class Config:
    vocab_size: int = 6564
    mel_channels: int = 80
    token_dim: int = 512
    speaker_dim: int = 256
    hidden_dim: int = 512
    nhead: int = 8
    lookahead: int = 5
    enc_layers: int = 2
    dit_layers: int = 6
    mlp_ratio: int = 4
    pad_token_id: int = 6563
    upsample_factor: int = 2
    max_mel_spectrogram_len: int = 1000

In [ ]:
FLOW_CONFIG = Config()

In [ ]:
class PovaryoshkaFlowDataset(Dataset):
    def __init__(
        self,
        data_list,
        cfg: Config | None = None
    ):
        self.cfg = cfg or Config()
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        mel_spectrogram = item["mel"]
        codes = item["codes"]
        speaker_embedding = item["speaker_embedding"]
        max_mel_spectrogram_len = self.cfg.max_mel_spectrogram_len
        if mel_spectrogram.shape[1] > max_mel_spectrogram_len:
            mel_spectrogram = mel_spectrogram[:, :max_mel_spectrogram_len]
            # Since the frame rate frequency is <upsample_factor> the code frequency
            codes = codes[:max_mel_spectrogram_len // self.cfg.upsample_factor]
        mel_spectrogram_length = mel_spectrogram.shape[1]
        return mel_spectrogram, codes, speaker_embedding, mel_spectrogram_length


In [ ]:
def get_padded_mel_spectrograms(mel_spectrograms):
    """Return: (B, 80, T_max)"""
    padded_mels = pad_sequence(
        [mel.transpose(0, 1) for mel in mel_spectrograms],
        batch_first=True,
        padding_value=0.0
    ).transpose(1, 2).contiguous()
    return padded_mels


def get_padded_codes_list(codes_list):
    """Return: (B, T_max)"""
    return pad_sequence(codes_list, batch_first=True, padding_value=FLOW_CONFIG.pad_token_id)

def collate_fn(batch):
    mel_spectrograms, codes_list, speaker_embeddings, mel_spectrogram_lengths = zip(*batch)
    padded_mel_spectrograms = get_padded_mel_spectrograms(mel_spectrograms)
    padded_codes_list = get_padded_codes_list(codes_list)
    speaker_embeddings = torch.stack(speaker_embeddings)
    return padded_mel_spectrograms, padded_codes_list, speaker_embeddings


In [ ]:
dummy_data = [None] * 100 
cfm_dataset = PovaryoshkaFlowDataset(dummy_data)
cfm_dataloader = DataLoader(
    cfm_dataset, 
    batch_size=32, 
    shuffle=True, 
    collate_fn=collate_fn
)

In [ ]:
log_dir = f"runs/cfm_pretrain_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir)
print(f"TensorBoard logs will be saved to: {log_dir}")

In [ ]:
def get_time_embedding(t, dim, scale=1000.0, max_period=10000.0):
    """Sin time embedding for continuous time t."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = (t.float() * scale)[:, None] * freqs[None, :]
    return torch.cat([args.sin(), args.cos()], dim=-1)


def build_rope_matrix_components(T, dim, device, base=10000.0):
    """
    Build the cosine and sine components for RoPE (Rotary Positional Embedding).
    Returns:
        cos: Tensor of shape (T, dim//2)
        sin: Tensor of shape (T, dim//2)
    """
    frequences = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    angles = torch.outer(torch.arange(T, device=device).float(), frequences)
    angles = torch.cat([angles, angles], dim=-1)
    return angles.cos(), angles.sin()
 
def apply_rope_matrix_components(embeddings, cos, sin):
    embeddings_part1, embeddings_part2 = embeddings.chunk(2, dim=-1)
    rotated_embeddings = torch.cat([-embeddings_part2, embeddings_part1], dim=-1)
    return embeddings * cos.to(embeddings.dtype) + rotated_embeddings * sin.to(embeddings.dtype)


def apply_rope_to_embeddings(embeddings):
    T = embeddings.size(1)
    hidden_dim = embeddings.size(2)
    cos, sin = build_rope_matrix_components(
        T=T,
        dim=hidden_dim,
        device=embeddings.device
    )
    roped_embeddings = apply_rope_matrix_components(embeddings, cos, sin)
    return roped_embeddings


# Error
# Chunk should be consistent with parameter M from speech synthesis
def build_chunk_aware_mask(lengths, T, chunk, device):
    """
    Return chunk aware mask (B,1,T,T), True = attention is allowed, False = attention is masked.
    chunk=0: full attention; chunk=N: token sees its own chunk and previous chunks.
    Padding tokens (>= length) is always forbidden.
    """
    idx = torch.arange(T, device=device)
    valid = idx[None, :] < lengths[:, None]                       # (B, T)
    if chunk > 0:
        c = idx // chunk
        struct = c[None, :] <= c[:, None]                         # (T, T)
    else:
        struct = torch.ones(T, T, dtype=torch.bool, device=device)
    return (struct[None] & valid[:, None, :])[:, None]

In [ ]:
class LookaheadPreConv(nn.Module):
    def __init__(self, cfg: Config | None = None):
        super().__init__()
        self.cfg = cfg or Config()
        self.conv = nn.Conv1d(
            in_channels=self.cfg.hidden_dim,
            out_channels=self.cfg.hidden_dim,
            kernel_size=self.cfg.lookahead + 1
        )

    def forward(self, x):
        h = F.pad(x.transpose(1, 2), (0, self.cfg.lookahead))
        return x + F.gelu(self.conv(h)).transpose(1, 2)

In [ ]:
class CausalUpsamplingTransformer(nn.Module):
    def __init__(self, cfg: Config | None = None):
        super().__init__()
        self.cfg = cfg or Config()
        self.pre_layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.hidden_dim,
            nhead=self.cfg.nhead,
            dim_feedforward=self.cfg.hidden_dim * self.cfg.mlp_ratio,
            batch_first=True,
            activation="gelu"
        )
        self.up_conv = nn.Conv1d(
            in_channels=self.cfg.hidden_dim,
            out_channels=self.cfg.hidden_dim,
            kernel_size=3,
            padding=1
        )
        self.post_layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.hidden_dim,
            nhead=self.cfg.nhead,
            dim_feedforward=self.cfg.hidden_dim * self.cfg.mlp_ratio,
            batch_first=True,
            activation="gelu"
        )

    def forward(
        self,
        speech_token_embeddings,
        src_speech_token_embedding_mask,
        src_upsampled_speech_token_embedding_mask
    ):
        x = self.pre_layer(speech_token_embeddings, src_mask=src_speech_token_embedding_mask)
        x = x.transpose(1, 2) # (B, C, T)
        x = F.interpolate(x, scale_factor=self.cfg.upsample_factor, mode="nearest")
        x = x.transpose(1, 2) + self.up_conv(x).transpose(1, 2) # (B, 2T, C)
        x = apply_rope_to_embeddings(x)
        x = self.post_layer(x, src_mask=src_upsampled_speech_token_embedding_mask)        
        return x


In [ ]:
class CausalTransformerEncoder(nn.Module):
    def __init__(self, cfg: Config | None = None):
        super().__init__()
        self.cfg = cfg or Config()
        layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.hidden_dim,
            nhead=self.cfg.nhead,
            dim_feedforward=self.cfg.hidden_dim * self.cfg.mlp_ratio,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=self.cfg.enc_layers)

    def forward(
        self,
        x,
        mask=None,
    ):
        return self.encoder(x, mask=mask)

In [ ]:
class DiTBlock(nn.Module):
    def __init__(self, cfg: Config | None = None):
        super().__init__()
        self.cfg = cfg or Config()
        self.norm1 = nn.LayerNorm(self.cfg.hidden_dim, elementwise_affine=False)
        self.attention = nn.MultiheadAttention(self.cfg.hidden_dim, self.cfg.nhead, batch_first=True)
        self.norm2 = nn.LayerNorm(self.cfg.hidden_dim, elementwise_affine=False)
        mlp_hidden = self.cfg.hidden_dim * self.cfg.mlp_ratio
        self.mlp = nn.Sequential(
            nn.Linear(self.cfg.hidden_dim, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, self.cfg.hidden_dim)
        )
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(self.cfg.hidden_dim, 6 * self.cfg.hidden_dim)
        )
        nn.init.zeros_(self.adaLN_modulation[1].weight) # type: ignore
        nn.init.zeros_(self.adaLN_modulation[1].bias) # type: ignore

    def forward(self, x, conditions, src_x_t_mask, x_t_rope_components):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
            self.adaLN_modulation(conditions).chunk(6, dim=1)
        x_norm = self.norm1(x) * (1 + scale_msa.unsqueeze(1)) + shift_msa.unsqueeze(1)
        x_norm = apply_rope_matrix_components(x_norm, cos=x_t_rope_components[0], sin=x_t_rope_components[1])
        attn_out, _ = self.attention(
            query=x_norm,
            key=x_norm,
            value=x_norm,
            attn_mask=src_x_t_mask
        )
        x = x + gate_msa.unsqueeze(1) * attn_out 
        x_norm = self.norm2(x) * (1 + scale_mlp.unsqueeze(1)) + shift_mlp.unsqueeze(1)
        return x + gate_mlp.unsqueeze(1) * self.mlp(x_norm)


class PovaryoshkaDiT(nn.Module):
    def __init__(self, cfg: Config | None = None):
        super().__init__()
        self.cfg = cfg or Config()
        self.mel_spectrogram_proj = nn.Linear(self.cfg.mel_channels, self.cfg.hidden_dim)
        self.token_embedding_proj = nn.Linear(self.cfg.hidden_dim, self.cfg.hidden_dim)
        self.speaker_embedding_proj = nn.Linear(self.cfg.speaker_dim, self.cfg.hidden_dim)
        self.time_embedding_proj = nn.Sequential(
            nn.Linear(self.cfg.hidden_dim, self.cfg.hidden_dim),
            nn.SiLU(),
            nn.Linear(self.cfg.hidden_dim, self.cfg.hidden_dim)
        )
        self.dit_blocks = nn.ModuleList([
            DiTBlock(self.cfg) for _ in range(self.cfg.dit_layers)
        ])
        self.final_norm = nn.LayerNorm(self.cfg.hidden_dim, elementwise_affine=False)
        self.final_linear = nn.Linear(self.cfg.hidden_dim, self.cfg.mel_channels)
        nn.init.zeros_(self.final_linear.weight)
        nn.init.zeros_(self.final_linear.bias)

    def forward(
        self,
        x_t, # (B, C, T)
        t,
        token_embeddings,
        speaker_embeddings,
        src_x_t_mask,
        x_t_rope_components
    ):
        x = self.mel_spectrogram_proj(x_t.transpose(1, 2)) + self.token_embedding_proj(token_embeddings)
        conditions = self.time_embedding_proj(get_time_embedding(t, self.cfg.hidden_dim)) + self.speaker_embedding_proj(speaker_embeddings)
        for block in self.dit_blocks:
            x = block(x, conditions, src_x_t_mask, x_t_rope_components)
        return self.final_linear(self.final_norm(x)).transpose(1, 2)


In [ ]:
class PovaryoshkaChunkAwareFlowMatch(nn.Module):
    def __init__(self, cfg: Config | None = None):
        super().__init__()
        self.cfg = cfg or Config()
        self.embed = nn.Embedding(
            self.cfg.vocab_size,
            self.cfg.hidden_dim,
            padding_idx=self.cfg.pad_token_id
        )
        self.lookahead_pre_conv = LookaheadPreConv(self.cfg)
        self.causal_upsampling_transformer = CausalUpsamplingTransformer(self.cfg)
        self.causal_transformer_encoder = CausalTransformerEncoder(self.cfg)
        self.povaryoshka_DiT = PovaryoshkaDiT(self.cfg)

    def forward(
        self,
        x_t_batch, # (B, C, T)
        t_batch,
        speech_tokens,
        speaker_embeddings
    ):
        speech_token_lengths = torch.sum(speech_tokens != self.cfg.pad_token_id, dim=1)
        T = speech_tokens.size(1)
        speech_token_embedings = self.embed(speech_tokens) # (B, T, hidden_dim)
        embeddings = self.lookahead_pre_conv(speech_token_embedings)
        src_embedding_mask = build_chunk_aware_mask(
            lengths=speech_token_lengths, 
            T=T, 
            chunk=self.cfg.lookahead, 
            device=speech_tokens.device
        )
        roped_embeddings = apply_rope_to_embeddings(embeddings)
        src_upsampled_embedding_mask = build_chunk_aware_mask(
            lengths=speech_token_lengths * self.cfg.upsample_factor, 
            T=T * self.cfg.upsample_factor, 
            chunk=self.cfg.lookahead * self.cfg.upsample_factor, 
            device=speech_tokens.device
        )
        upsampled_embeddings = self.causal_upsampling_transformer(
            roped_embeddings,
            src_embedding_mask,
            src_upsampled_embedding_mask
        )
        upsampled_roped_embeddings = apply_rope_to_embeddings(upsampled_embeddings)
        upsampled_speech_token_embeddings = self.causal_transformer_encoder(
            upsampled_roped_embeddings,
            src_upsampled_embedding_mask
        )
        x_t_rope_components = build_rope_matrix_components(
            T=T * self.cfg.upsample_factor,
            dim=self.cfg.hidden_dim,
            device=x_t_batch.device
        )
        return self.povaryoshka_DiT(
            x_t_batch,
            t_batch,
            upsampled_speech_token_embeddings,
            speaker_embeddings,
            src_x_t_mask=src_upsampled_embedding_mask,
            x_t_rope_components=x_t_rope_components
        )

In [ ]:
@torch.no_grad()
def sample(model, mu, v_speaker, T, C_mel, num_steps=10, cfg_scale=1.5, device='cuda'):
    """
    Генерация аудио (ODE solver - Euler).
    """
    # Стартуем с шума
    x = torch.randn(1, C_mel, T, device=device)
    
    # Время шага (косинусный планировщик, Формула 9)
    def get_cosine_t(step):
        return 1 - torch.cos(torch.tensor((step / num_steps) * (torch.pi / 2)))
    
    # В статье: "train on both conditional and non-conditional to enable CFG"
    # null_cond_mu и null_v - это пустые тензоры (условно нули)
    null_mu = torch.zeros_like(mu)
    null_v = torch.zeros_like(v_speaker)
    
    for step in range(num_steps):
        t_curr = get_cosine_t(step)
        t_next = get_cosine_t(step + 1)
        dt = t_next - t_curr
        
        t_tensor = torch.tensor([t_curr], device=device)
        
        # Предсказание с условиями (conditional)
        v_cond = model(x, t_tensor, mu, x, v_speaker)
        
        # Предсказание без условий (unconditional)
        v_uncond = model(x, t_tensor, null_mu, x, null_v)
        
        # Формула 10: Classifier-Free Guidance
        v_t = (1 + cfg_scale) * v_cond - cfg_scale * v_uncond
        
        # Шаг ODE (Euler)
        x = x + v_t * dt
        
    return x

In [ ]:
def get_phi_t_and_u_t(t, noise, target_mel_spectrograms):
    t_expand = t[:, None, None]
    phi_t = (1 - t_expand) * noise + t_expand * target_mel_spectrograms
    u_t = target_mel_spectrograms - noise
    return phi_t, u_t


def CFM_train_step(
    model,
    optimizer,
    scheduler,
    target_mel_spectrograms,
    token_embeddings,
    speaker_embeddings,
    device
):
    B, _, T = target_mel_spectrograms.shape
    t = torch.rand(B, device=device)
    noise = torch.randn_like(target_mel_spectrograms)
    phi_t, u_t = get_phi_t_and_u_t(t, noise, target_mel_spectrograms)
    if torch.rand(1).item() < 0.1:
        token_embeddings = torch.zeros_like(token_embeddings)
        speaker_embeddings = torch.zeros_like(speaker_embeddings)
    v_t_pred = model(phi_t, t, token_embeddings, speaker_embeddings)
    loss = F.mse_loss(v_t_pred, u_t)
    optimizer.zero_grad()
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()
    optimizer.step()
    scheduler.step()
    return loss.item(), grad_norm

In [ ]:
def CFM_train_loop(
    model, 
    dataloader, 
    optimizer, 
    scheduler,
    device, 
    num_epochs=5,
    writer=None,
    log_every_steps=50
):
    model.train()
    global_step = 0
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch_idx, (target_mel_spectrograms, token_embeddings, speaker_embeddings) in enumerate(dataloader):
            target_mel_spectrograms = target_mel_spectrograms.to(device)
            token_embeddings = token_embeddings.to(device)
            speaker_embeddings = speaker_embeddings.to(device)
            loss_val, grad_norm = CFM_train_step(model, optimizer, scheduler, target_mel_spectrograms, token_embeddings, speaker_embeddings, device)
            epoch_loss += loss_val
            global_step += 1
            if writer is not None:
                writer.add_scalar("Train/Loss_step", loss_val, global_step)
                writer.add_scalar("Train/Grad_Norm", grad_norm, global_step)
                writer.add_scalar("Train/LR", scheduler.get_last_lr()[0], global_step)
            if batch_idx % log_every_steps == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Epoch {epoch} | Step {batch_idx} | Loss: {loss_val:.4f} | LR: {current_lr:.6f}")
        avg_loss = epoch_loss / len(dataloader)
        print(f"--- Epoch {epoch} finished. Avg Loss: {avg_loss:.4f} ---")
        if writer is not None:
            writer.add_scalar("Train/Loss_epoch", avg_loss, epoch)
    if writer is not None:
        writer.close()


In [ ]:
num_training_steps = len(cfm_dataloader) * 5 
num_warmup_steps = 500
model = PovaryoshkaChunkAwareFlowMatch().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)
CFM_train_loop(model, cfm_dataloader, optimizer, scheduler, device, num_epochs=5)